In [ ]:
import inspect
from groq import Groq
import json , re
api = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
model = "openai/gpt-oss-20b"
groq = Groq(api_key=api)

In [ ]:


# ---------------------------------------------------------------
# Tool registry: @tool registers the function AND builds its JSON schema
# (name / docstring / parameters) for Groq's native tool calling.
# ---------------------------------------------------------------
TOOLS = {}
TOOL_SCHEMAS = []

TYPE_MAP = {str: "string", int: "integer", float: "number", bool: "boolean"}


def tool(func):
    props, required = {}, []
    for pname, param in inspect.signature(func).parameters.items():
        props[pname] = {"type": TYPE_MAP.get(param.annotation, "string")}
        required.append(pname)

    TOOLS[func.__name__] = func
    TOOL_SCHEMAS.append(
        {
            "type": "function",
            "function": {
                "name": func.__name__,
                "description": inspect.getdoc(func),
                "parameters": {
                    "type": "object",
                    "properties": props,
                    "required": required,
                },
            },
        }
    )
    return func


@tool
def database(name: str):
    """Search the database by name and return a dictionary with the person's info (name, age, major, address)."""
    data = {
        "yousef": {"name": "yousef", "age": 25, "major": "ai", "address": "cairo"},
        "ali": {"name": "ali", "age": 30, "major": "frontend", "address": "cairo"},
        "sama": {"name": "sama", "age": 27, "major": "data analysis", "address": "cairo"},
    }
    if name not in data:
        raise KeyError(f"'{name}' not found in database")
    return data[name]


@tool
def write_file(content: str):
    """Write the given text to a text file (final.txt)."""
    if isinstance(content, dict):
        content = "\n".join(f"{k}: {v}" for k, v in content.items())
    with open("final.txt", mode="w", encoding="utf-8") as file:
        file.write(str(content))
    return "final.txt written successfully"


# ---------------------------------------------------------------
# ReAct prompt (tools are passed through the API, not described in text)
# ---------------------------------------------------------------
SYSTEM_PROMPT = """
You are a ReAct agent. You solve the user's request step by step using tools.

Loop:
1. Thought: before every tool call, write ONE short sentence explaining what
   you know and why you are calling that tool.
2. Action: call exactly ONE tool.
3. Observation: you will receive the real result of the tool. Use it.

Rules:
- Use real data from observations. Never invent data that should come from a tool.
- When you call write_file, copy the real data from the observations into "content".
- If a tool returns an error, think about what went wrong and try another action.
- When the whole request is done, reply with "Final Answer: ..." and do not call any tool.
"""


def run_tool(name: str, raw_arguments: str) -> str:
    if name not in TOOLS:
        return f"Error: unknown tool '{name}'. Available tools: {list(TOOLS)}"
    try:
        arguments = json.loads(raw_arguments or "{}")
        return json.dumps(TOOLS[name](**arguments), ensure_ascii=False)
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


# ---------------------------------------------------------------
# ReAct loop with native tool calling
# ---------------------------------------------------------------
def react_agent(question: str, max_steps: int = 3):
    print("=" * 60)
    print("ReAct Agent (native tool calling)")
    print(f"QUESTION: {question}")
    print("=" * 60)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        print(f"\n--- Step {step} ---")

        try:
            response = groq.chat.completions.create(
                model=model,
                messages=messages,
                tools=TOOL_SCHEMAS,
                tool_choice="auto",
                temperature=0,
            )
        except BadRequestError as e:
            # gpt-oss sometimes produces a malformed tool call -> ask it to retry
            print(f"[WARN] malformed tool call, retrying: {e}")
            messages.append(
                {
                    "role": "user",
                    "content": "Your last tool call was malformed. Call the tool again "
                    "with valid arguments.",
                }
            )
            continue

        msg = response.choices[0].message

        reasoning = getattr(msg, "reasoning", None)
        if reasoning:
            print(f"[REASONING] {reasoning.strip()}")
        if msg.content:
            print(msg.content.strip())

        # No tool call -> the model is giving its final answer
        if not msg.tool_calls:
            text = (msg.content or "").strip()
            return text.split("Final Answer:", 1)[-1].strip()

        # Keep the assistant turn (with its tool calls) in the history
        messages.append(
            {
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": call.id,
                        "type": "function",
                        "function": {
                            "name": call.function.name,
                            "arguments": call.function.arguments,
                        },
                    }
                    for call in msg.tool_calls
                ],
            }
        )

        # Act + Observe
        for call in msg.tool_calls:
            print(f"[ACTION] {call.function.name}({call.function.arguments})")
            observation = run_tool(call.function.name, call.function.arguments)
            print(f"[OBSERVATION] {observation}")
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": observation,
                }
            )

    return f"Stopped: reached the limit of {max_steps} steps without a final answer."


if __name__ == "__main__":
    q = "search by name = 'yousef' in database and create text file and write in his information"
    print("\nFINAL ANSWER:\n" + react_agent(q))